# IS630: Green Light for an Indie Studio
## Notebook 00: Setup, Acquisition, Join and Train/Test Split
**Group**: [Insert names]

> **AI Coding Assistant Declaration**:
> In accordance with course guidelines, an AI coding assistant was used strictly for code scaffolding and data preparation. All code blocks below are labeled as AI-assisted. All modelling decisions, interpretations, and written commentaries are independently authored by group members in the designated sections below.

| Dataset | Kaggle ref | Version | Pulled |
|---|---|---|---|
| Steam | `fronkongames/steam-games-dataset` | v49 | 19 Sep 2026 |
| HowLongToBeat | `b4n4n4p0wer/how-long-to-beat-video-game-playtime-dataset` | v5 | 19 Sep 2026 |

The Steam dataset refreshes continuously. Do not re-download mid-project.

**Reads:** `data/raw/steam_v49.zip`, `data/raw/hltb_v5.zip`
**Writes:** `00_steam.parquet`, `00_hltb.parquet`, `00_joined.parquet`, `00_train.parquet`, `00_test_holdout.parquet`, all in `data/interim`

## 1. Environment setup and data acquisition
*Code: Generated by AI Assistant*

In [1]:
import sys
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists()) / "src"))
from is630 import config
from is630.clean import normalise_title

TEST_FRACTION = 0.20

In [2]:
# Download only if the pinned archive is missing
for path, ref in {config.STEAM_ZIP: config.STEAM_REF, config.HLTB_ZIP: config.HLTB_REF}.items():
    if not path.exists():
        urllib.request.urlretrieve(f"https://www.kaggle.com/api/v1/datasets/download/{ref}", path)
    print(path.name, f"{path.stat().st_size / 1e6:.1f} MB")

steam_v49.zip 315.9 MB
hltb_v5.zip 18.1 MB


## 2. Steam dataset
*Code: Generated by AI Assistant*

*Written by AI Assistant*: the CSV header fuses `Discount` and `DLC count` into one name, giving 39 headers for 40 fields. Reading it as-is shifts every column one place, so the header is split before loading.

In [3]:
with zipfile.ZipFile(config.STEAM_ZIP) as z:
    with z.open("games.csv") as f:
        header = pd.read_csv(f, nrows=0).columns.tolist()

i = header.index("DiscountDLC count")
names = header[:i] + ["Discount", "DLC count"] + header[i + 1:]

with zipfile.ZipFile(config.STEAM_ZIP) as z:
    with z.open("games.csv") as f:
        steam = pd.read_csv(f, header=0, names=names, index_col=False)

print(f"steam: {steam.shape[0]:,} rows, {steam.shape[1]} columns")
steam.head()

steam: 125,855 rows, 40 columns


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,About the game,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",...,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
4,3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,...,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN


In [15]:
print("Missing values, columns with any:")
print(steam.isna().sum()[lambda v: v > 0].sort_values(ascending=False))
print("\nZero-filled rather than null:")
for col in ["Metacritic score", "Price", "Positive", "Negative", "Median playtime forever"]:
    print(f"  {col}: {(steam[col] == 0).sum():,} zeros")

steam.info()

Missing values, columns with any:
Movies            125855
Score rank        125815
Metacritic url    121597
Reviews           113674
Notes             102545
Website            75335
Support url        70560
Tags               42502
Support email      22630
Categories          8966
Publishers          8922
About the game      8462
Developers          8449
Genres              8423
Screenshots         6027
Header image          81
Name                   1
dtype: int64

Zero-filled rather than null:
  Metacritic score: 121,597 zeros
  Price: 26,661 zeros
  Positive: 45,509 zeros
  Negative: 58,432 zeros
  Median playtime forever: 99,682 zeros
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125855 entries, 0 to 125854
Data columns (total 42 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       125855 non-null  int64  
 1   Name                        125854 non-null  object 
 2   Rel

### Interpretation: Steam data inspection
**Author**: Lee Sin Yang 
<!-- Record your observations on the Steam table: row and column counts, which columns are missing values and whether that missingness looks random, which columns are zero-filled rather than null and what that hides, and which columns the three analytical questions will actually need. -->
Steam df has 42 columns and 125855 rows. Missingness varies across columns, with some columns such as the publishers, developers and about the game in the thousands, while movies, metacritic url and reviews has high missingness. This is likely due to most games not actually being very popular. 

## 3. HowLongToBeat dataset
*Code: Generated by AI Assistant*

In [5]:
with zipfile.ZipFile(config.HLTB_ZIP) as z:
    with z.open("hltb_dataset_normalized.csv") as f:
        hltb = pd.read_csv(f)

print(f"hltb: {hltb.shape[0]:,} rows, {hltb.shape[1]} columns")
hltb.head()

hltb: 166,754 rows, 28 columns


,id,name,type,platform,genres,developer,publisher,release_date,release_precision,release_year,...,all_styles_polled,all_styles,single_player_polled,single_player,co_op_polled,co_op,versus_polled,versus,source_url,crawled_at
0,1,688(I) Hunter/Killer,game,PC,"Simulation, Strategy/Tactical",Sonalysts,Electronic Arts,1997-07-04,day,1997.0,...,15.0,15.93,8.0,10.88,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/1,2026-03-27T10:06:13Z
1,2,Beyond Good & Evil 2,game,NaN,"Action, Adventure",Ubisoft Montpellier,Ubisoft,0000-00-00,day,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/2,2026-03-27T10:06:14Z
2,18,'Splosion Man,game,Xbox 360,Action,Twisted Pixel Games,Microsoft Games Studios,2009-07-22,day,2009.0,...,62.0,8.76,32.0,7.80,1.0,9.0,NaN,NaN,https://howlongtobeat.com/game/18,2026-03-27T10:06:16Z
3,19,.hack//G.U. Vol. 1: Rebirth,game,"Nintendo Switch, PC, PlayStation 2, PlayStation 4","Action, Role-Playing",CyberConnect2,Bandai,2006-05-18,day,2006.0,...,181.0,23.41,76.0,18.52,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/19,2026-03-27T10:06:15Z
4,20,.hack//G.U. Vol. 2: Reminisce,game,"Nintendo Switch, PC, PlayStation 2, PlayStation 4","Action, Role-Playing",CyberConnect2,Bandai,2006-09-28,day,2006.0,...,88.0,26.62,41.0,19.82,NaN,NaN,NaN,NaN,https://howlongtobeat.com/game/20,2026-03-27T10:06:15Z


In [16]:
print("Entry types:")
print(hltb["type"].value_counts())
print(f"\nmain_story missing: {hltb['main_story'].isna().sum():,} of {len(hltb):,}")
print(f"release_year missing: {hltb['release_year'].isna().sum():,} of {len(hltb):,}")

hltb.info()

Entry types:
type
game             155586
dlc/expansion      2728
omit               2587
multi              2249
delay               894
compil              691
sports              645
mod                 516
endless             454
hack                404
Name: count, dtype: int64

main_story missing: 120,300 of 166,754
release_year missing: 70,593 of 166,754
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166754 entries, 0 to 166753
Data columns (total 29 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      166754 non-null  int64  
 1   name                    166754 non-null  object 
 2   type                    166754 non-null  object 
 3   platform                91397 non-null   object 
 4   genres                  82852 non-null   object 
 5   developer               89017 non-null   object 
 6   publisher               85275 non-null   object 
 7   release_date            96161 non-

### Interpretation: HowLongToBeat data inspection
**Author**: Lee Sin Yang 
<!-- Record your observations on the HowLongToBeat table: how many entries are games rather than DLC or other types, how much of the table carries a main-story time, and what that implies for how many Steam games can receive a completion ratio. -->
The HLTB dataset has 29 columns and 166754 rows. Missingness varies across columns, as many fields are optional and only applicable to certain types of games. all_styles_polled, our candidate target variable, has about 57K non null rows.

## 4. Save the raw tables
*Code: Generated by AI Assistant*

*Written by AI Assistant*: both tables are saved as loaded. Column pruning and type cleaning belong to later stages.

In [7]:
steam.to_parquet(config.INTERIM / "00_steam.parquet", index=False)
hltb.to_parquet(config.INTERIM / "00_hltb.parquet", index=False)
print(f"saved unmodified: {len(steam):,} steam rows, {len(hltb):,} hltb rows")

saved unmodified: 125,855 steam rows, 166,754 hltb rows


## 5. Joining the two datasets
*Code: Generated by AI Assistant*

*Written by AI Assistant*: no shared identifier exists, so the key is the normalised title. Release year is recorded rather than used as part of the key, because HowLongToBeat stores a game's original release year while Steam stores its Steam release date. Requiring the two to agree would drop console-first ports and re-releases, which are among the best covered titles.

Both sides are reduced to one row per title first, so the join cannot multiply rows:

- HowLongToBeat is filtered to `type == "game"` with a positive `main_story`, then for each title the candidate whose release year is closest to the Steam release year is kept, breaking ties on the larger `main_story_polled`.
- Steam keeps, per title, the entry with the most reviews.

The join is a left join, so every Steam game survives and `matched_hltb` marks the ones that found a partner. `year_diff` is the Steam year minus the HowLongToBeat year, for tightening the match later.

In [8]:
steam["title_key"] = normalise_title(steam["Name"])
steam["steam_year"] = pd.to_datetime(steam["Release date"], errors="coerce", format="mixed").dt.year

hltb["title_key"] = normalise_title(hltb["name"])

candidates = hltb[hltb["type"].eq("game") & hltb["main_story"].notna() & hltb["main_story"].gt(0)].copy()
print(f"hltb candidates with a main story time: {len(candidates):,}")

steam_one = (
    steam.assign(_reviews=steam["Positive"] + steam["Negative"])
    .sort_values("_reviews", ascending=False)
    .drop_duplicates("title_key")
    .drop(columns="_reviews")
)
print(f"steam rows dropped as duplicate titles: {len(steam) - len(steam_one):,}")

hltb candidates with a main story time: 42,167
steam rows dropped as duplicate titles: 4,571


In [9]:
# Pick, per title, the hltb candidate closest in year to the Steam release
pairs = steam_one[["title_key", "steam_year"]].merge(candidates, on="title_key", how="inner")
pairs["_gap"] = (pairs["steam_year"] - pairs["release_year"]).abs()

hltb_one = (
    pairs.sort_values(["_gap", "main_story_polled"], ascending=[True, False])
    .drop_duplicates("title_key")
    .drop(columns=["_gap", "steam_year"])
)

joined = steam_one.merge(hltb_one, on="title_key", how="left", suffixes=("", "_hltb"))
joined["matched_hltb"] = joined["main_story"].notna()
joined["year_diff"] = joined["steam_year"] - joined["release_year"]

assert len(joined) == len(steam_one), "join multiplied rows"
matched = int(joined["matched_hltb"].sum())
agree = int((joined["year_diff"].abs() <= 1).sum())
print(f"rows in: {len(steam_one):,}")
print(f"matched to hltb: {matched:,} ({matched / len(joined) * 100:.1f}%)")
print(f"  of which year agrees within 1: {agree:,} ({agree / matched * 100:.1f}% of matches)")
print(f"rows out: {len(joined):,}")

rows in: 121,284
matched to hltb: 20,826 (17.2%)
  of which year agrees within 1: 17,845 (85.7% of matches)
rows out: 121,284


In [10]:
# Matched against unmatched, on the variables both groups have
compare = joined.groupby("matched_hltb").agg(
    games=("AppID", "size"),
    median_reviews=("Positive", "median"),
    median_price=("Price", "median"),
    pct_with_metacritic=("Metacritic score", lambda s: (s > 0).mean() * 100),
    median_year=("steam_year", "median"),
)
compare.round(2)

,games,median_reviews,median_price,pct_with_metacritic,median_year
matched_hltb,,,,,
False,100458,2.0,1.99,0.81,2023.0
True,20826,155.0,3.49,16.19,2020.0


In [11]:
joined.to_parquet(config.INTERIM / "00_joined.parquet", index=False)
joined[["AppID", "Name", "steam_year", "release_year", "year_diff", "main_story", "main_story_polled", "matched_hltb"]].head()

,AppID,Name,steam_year,release_year,year_diff,main_story,main_story_polled,matched_hltb
0,730,Counter-Strike 2,2012,NaN,NaN,NaN,NaN,False
1,578080,PUBG: BATTLEGROUNDS,2017,NaN,NaN,NaN,NaN,False
2,570,Dota 2,2013,NaN,NaN,NaN,NaN,False
3,271590,Grand Theft Auto V Legacy,2015,NaN,NaN,NaN,NaN,False
4,105600,Terraria,2011,2011.0,0.0,51.74,689.0,True


In [14]:
joined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121284 entries, 0 to 121283
Data columns (total 72 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       121284 non-null  int64  
 1   Name                        121284 non-null  object 
 2   Release date                121284 non-null  object 
 3   Estimated owners            121284 non-null  object 
 4   Peak CCU                    121284 non-null  int64  
 5   Required age                121284 non-null  int64  
 6   Price                       121284 non-null  float64
 7   Discount                    121284 non-null  int64  
 8   DLC count                   121284 non-null  int64  
 9   About the game              113369 non-null  object 
 10  Supported languages         121284 non-null  object 
 11  Full audio languages        121284 non-null  object 
 12  Reviews                     12060 non-null   object 
 13  Header image  

### Interpretation: The join and its match rate
**Author**: Lee Sin Yang
<!-- Interpret the match rate and the matched-versus-unmatched comparison above. Cover whether the matched subset looks representative of the Steam population, what the direction of any difference implies for the completion ratio results, whether the normalised-title key should be tightened to require the release years to agree, and how this should be reported as a limitation. -->
The match rate is fairly low, due to the long tail of steam games by reviews. Most games on steam have little to no reviews, and thus the ones that survive tend to be more popular. This is an important consideration for future analyses - our data only captures games that perform well enough to appear on HLTB, a form of survivorship bias. 

## 6. Train and test split
*Code: Generated by AI Assistant*

*Written by AI Assistant*: the test set is held out now, before any exploration, and is scored once at the end. Nothing is refitted after that score.

The split is stratified on release year crossed with `matched_hltb`, so both sets span the same years and carry the same share of matched games. A plain random split would leave the match rate and the age mix differing between them, and review counts and playtime both accumulate with age.

A time-based holdout, training on older games and testing on the newest, would be the stricter test of predicting an unreleased game. It is not used here because every accumulating column would then differ systematically between train and test, which would confound the score.

In [12]:
strata = joined["matched_hltb"].astype(str) + "_" + joined["steam_year"].fillna(-1).astype(int).astype(str)
rare = strata.value_counts()[lambda v: v < 5].index
strata = strata.where(~strata.isin(rare), "rare")

test = joined.groupby(strata, group_keys=False).sample(frac=TEST_FRACTION, random_state=config.SEED)
train = joined.drop(index=test.index)

assert len(train) + len(test) == len(joined)
assert not set(train["AppID"]) & set(test["AppID"])

for name, df in [("00_train", train), ("00_test_holdout", test)]:
    df.to_parquet(config.INTERIM / f"{name}.parquet", index=False)
    print(f"{name}: {len(df):,} rows ({len(df) / len(joined) * 100:.1f}%), matched {df['matched_hltb'].mean() * 100:.1f}%")

00_train: 97,026 rows (80.0%), matched 17.2%
00_test_holdout: 24,258 rows (20.0%), matched 17.2%


*Written by AI Assistant*: `00_test_holdout.parquet` must not be loaded by any notebook except the final scoring step. Every later stage starts from `00_train.parquet`.